<a href="https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Setup

**Run this before the lab.** It checks that your LangSmith key works, that model calls go
through, and that traces land in **your own workspace**.

You need:

- A **personal Google account**, to run Colab.
- Your **LangSmith invite** accepted. It went to your work email.

---

## Step 1 — Open your workspace

Open [smith.langchain.com](https://smith.langchain.com) and sign in with your work email. Use
the workspace switcher at the top left and pick the workspace with **your name**.

---

## Step 2 — Create your API key

1. In LangSmith, open **Settings** → **API Keys**.
2. Create a new API key of type **Personal Access Token**.
3. Name it `hse workshop` and set it to expire in **7 days**.
4. Create it and **copy the key**. LangSmith shows it only once.

---

## Step 3 — Store your key in Colab Secrets

1. In the **left sidebar of this notebook**, click the 🔑 key icon.
2. Click **Add new secret**.
3. Name it **exactly** `LANGSMITH_API_KEY`.
4. Paste your key into **Value**.
5. **Turn on "Notebook access".**

### Running locally instead?

From a clone of this repo, `uv sync`, copy `.env.example` to `.env`, paste your key, then
`uv run jupyter lab notebooks`. The setup cell reads `.env` when there is no Colab.

---

## Step 4 — Run the setup cell

Run the cell. It finds your workspace from your key. If it asks for one, type your workspace
name in the **WORKSPACE** field and run it again.

In [ ]:
# --- snippet:setup v1 ---
import os

import requests

WORKSPACE = ""  #@param {type:"string"}

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata:
    %pip install -qq --progress-bar off \
      "langchain~=1.4.3" \
      "langchain-openai~=1.6.7" \
      "langsmith~=0.14.4" \
      "agentevals~=0.0.9"

    try:
        os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise RuntimeError(
            "Add LANGSMITH_API_KEY in the 🔑 Secrets panel, turn on Notebook access, "
            "then run this cell again."
        ) from None
else:
    from dotenv import load_dotenv

    load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "hse-00-setup"

endpoint = os.environ.get("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
reply = requests.get(
    f"{endpoint}/api/v1/workspaces",
    headers={"x-api-key": os.environ["LANGSMITH_API_KEY"]},
    timeout=30,
)
if reply.status_code in (401, 403):
    raise RuntimeError("LangSmith rejected your key. Create a new one and paste it into 🔑 Secrets.")
reply.raise_for_status()
workspaces = {w["id"]: w["display_name"] for w in reply.json()}

# Pin the workspace: a key that reaches several sends everything to its default one, which may
# not be yours.
if WORKSPACE:
    wanted = WORKSPACE.strip().casefold()
    matches = [id_ for id_, name in workspaces.items() if name.casefold() == wanted]
    if len(matches) != 1:
        raise ValueError(f"No workspace named {WORKSPACE!r}. Type it exactly as LangSmith shows it.")
    os.environ["LANGSMITH_WORKSPACE_ID"] = matches[0]
elif len(workspaces) == 1:
    os.environ["LANGSMITH_WORKSPACE_ID"] = next(iter(workspaces))
elif os.environ.get("LANGSMITH_WORKSPACE_ID") not in workspaces:
    raise ValueError("Type your workspace name in the WORKSPACE field, then run this cell again.")

MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready. Workspace:", workspaces[os.environ["LANGSMITH_WORKSPACE_ID"]])

---

## Step 5 — Test a model call

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL)

response = model.invoke("Return the text 'Hello, World!'.")
response.text

---

## Step 6 — Check the trace landed in your workspace

Open the link. It should show the model call from Step 5, in the workspace with your name.

In [ ]:
import asyncio

from langsmith import Client
from langsmith.utils import LangSmithNotFoundError

client = Client()

for _ in range(10):
    try:
        project = client.read_project(project_name=os.environ["LANGSMITH_PROJECT"])
        traces = await client.runs.query(
            project_ids=[str(project.id)], is_root=True, page_size=1, selects=["ID", "TRACE_ID"]
        )
        if traces.items:
            break
    except LangSmithNotFoundError:
        pass
    await asyncio.sleep(3)
else:
    raise RuntimeError("No trace yet. Wait a few seconds and run this cell again.")

trace = traces.items[0]
link = await client.runs.get_url(
    str(trace.id), project_id=str(project.id), trace_id=str(trace.trace_id)
)
print("Your trace:", link.url)

---

## ➡️ Next

**[01 · Find the bugs, then prove the fix](https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/01_find_and_fix.ipynb)**